# Apply Excel Re-ID Mapping

Reads a manually-edited Excel ID mapping and applies it to a tracker CSV.

**Excel format expected:**
- Sheet `Team P0` — columns = players, rows = tracker IDs. First row = canonical ID, rest are aliases (no frame constraint).
- Sheet `Team P1` — alternating (ID, frame) column pairs. First ID = canonical, rest are aliases with optional frame range (e.g. `4270-5750`, `4632-`, `5128`).

**Rules applied:**
- Data before `MIN_FRAME` is never touched.
- If an alias ID is also a canonical elsewhere, it is skipped (conflict protection).
- Frame ranges have ±`TOLERANCE` frames tolerance.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
TRACKS_CSV  = "/content/per_frame_tracks_HILHAZ_half1.csv"   # input tracker CSV
EXCEL_FILE  = "/content/half1_edited_v2.xlsx"                # your Excel mapping
OUTPUT_CSV  = "/content/per_frame_tracks_half1_reid.csv"     # output

MIN_FRAME   = 4270   # ignore all frames below this
TOLERANCE   = 10     # ±frames around frame-range endpoints

In [ ]:
# ── Install deps if needed ────────────────────────────────────────────────────
import subprocess, sys
for pkg in ["pandas", "openpyxl"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Setup OK.")

In [ ]:
# ── Apply Excel Re-ID mapping ─────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re

# ── helpers ──────────────────────────────────────────────────────────────────
def parse_id_cell(val):
    """Returns (tracker_id, class_filter) or (None, None).
    class_filter: 1=GK, 2=player, 3=referee, None=any.
    Handles: '20', '1 REF', '2 P', 'GK1', '25 PT1' (PT = team note, ignored).
    """
    if pd.isna(val): return None, None
    s = str(val).strip()
    if not s or s.lower() == 'nan': return None, None
    m = re.match(r'^GK\s*(\d+)', s, re.IGNORECASE)
    if m: return int(m.group(1)), 1
    m = re.match(r'^(\d+)\s*(.*?)$', s)
    if not m: return None, None
    tid = int(m.group(1))
    ann = m.group(2).strip().upper()
    if 'REF' in ann: return tid, 3
    if ann == 'P':   return tid, 2
    if 'GK' in ann:  return tid, 1
    return tid, None

def parse_frame_range(val):
    """Returns (start, end) where None means unbounded."""
    if pd.isna(val): return None, None
    s = str(val).strip()
    if not s or s.lower() == 'nan': return None, None
    if '-' in s:
        parts = s.split('-')
        try: start = int(parts[0]) if parts[0].strip() else None
        except: start = None
        try: end   = int(parts[1]) if len(parts) > 1 and parts[1].strip() else None
        except: end = None
        return start, end
    try: f = int(s); return f, f
    except: return None, None

# ── collect all canonical IDs ─────────────────────────────────────────────────
canonical_ids = set()
p0 = pd.read_excel(EXCEL_FILE, sheet_name='Team P0', header=None)
for col_idx in range(p0.shape[1]):
    for r in range(1, p0.shape[0]):
        tid, _ = parse_id_cell(p0.iloc[r, col_idx])
        if tid is not None:
            canonical_ids.add(tid); break

p1 = pd.read_excel(EXCEL_FILE, sheet_name='Team P1', header=None)
for i in range(0, p1.shape[1], 2):
    for r in range(1, p1.shape[0]):
        tid, _ = parse_id_cell(p1.iloc[r, i])
        if tid is not None:
            canonical_ids.add(tid); break

print(f"Canonical IDs ({len(canonical_ids)}): {sorted(canonical_ids)}")

# ── build remap rules ─────────────────────────────────────────────────────────
rules    = []
used_src = {}  # src_id → dst_id (first-assignment wins)
skipped  = []

def add_rule(src_id, src_cf, dst_id, fstart, fend, origin):
    if src_id in canonical_ids and src_id != dst_id:
        skipped.append(f"{src_id}→{dst_id} ({origin}): {src_id} is canonical")
        return
    if src_id in used_src and used_src[src_id] != dst_id:
        skipped.append(f"{src_id}→{dst_id} ({origin}): already mapped to {used_src[src_id]}")
        return
    used_src[src_id] = dst_id
    rules.append({'src_id': src_id, 'src_cf': src_cf, 'dst_id': dst_id,
                  'fstart': fstart, 'fend': fend})

for col_idx in range(p0.shape[1]):
    vals = []
    for r in range(1, p0.shape[0]):
        tid, cf = parse_id_cell(p0.iloc[r, col_idx])
        if tid is not None: vals.append((tid, cf))
    if not vals: continue
    canon_id = vals[0][0]
    for (tid, cf) in vals[1:]:
        add_rule(tid, cf, canon_id, None, None, f'P0col{col_idx}')

for i in range(0, p1.shape[1], 2):
    if i+1 >= p1.shape[1]: break
    pairs = []
    for r in range(1, p1.shape[0]):
        tid, cf   = parse_id_cell(p1.iloc[r, i])
        fs, fe    = parse_frame_range(p1.iloc[r, i+1])
        if tid is not None: pairs.append({'id': tid, 'cf': cf, 'fstart': fs, 'fend': fe})
    if not pairs: continue
    canon_id = pairs[0]['id']
    for p in pairs[1:]:
        add_rule(p['id'], p['cf'], canon_id, p['fstart'], p['fend'], f'P1col{i//2}')

print(f"Rules applied: {len(rules)}  |  Skipped (conflicts): {len(skipped)}")
for r in rules:
    print(f"  {r['src_id']}(class={r['src_cf']}) → {r['dst_id']}  "
          f"frames=[{r['fstart']},{r['fend']}]")

# ── load CSV ──────────────────────────────────────────────────────────────────
df = pd.read_csv(TRACKS_CSV, encoding='utf-8', encoding_errors='replace',
                 low_memory=False)
print(f"\nCSV: {len(df):,} rows  |  frames {df['frame'].min()}–{df['frame'].max()}")

orig_ids     = df['display_track_id'].copy()
total_changed = 0

for rule in rules:
    m = ((df['frame'] >= MIN_FRAME) &
         (df['display_track_id'] == rule['src_id']))
    if rule['src_cf'] is not None:
        m &= (df['class_id'] == rule['src_cf'])
    if rule['fstart'] is not None:
        m &= (df['frame'] >= rule['fstart'] - TOLERANCE)
    if rule['fend'] is not None:
        m &= (df['frame'] <= rule['fend'] + TOLERANCE)
    n = m.sum()
    if n > 0:
        df.loc[m, 'display_track_id'] = rule['dst_id']
        total_changed += n
        print(f"  {rule['src_id']}→{rule['dst_id']}: {n} rows")

print(f"\nRows changed: {total_changed}  |  "
      f"Untouched (frames <{MIN_FRAME}): {(df['frame'] < MIN_FRAME).sum()}")

assert ((df['display_track_id'] != orig_ids) & (df['frame'] < MIN_FRAME)).sum() == 0, \
    "BUG: data before MIN_FRAME was modified!"

# ── save ──────────────────────────────────────────────────────────────────────
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved → {OUTPUT_CSV}")
print("\nID counts (frames ≥ MIN_FRAME, people only):")
after = df[(df['frame'] >= MIN_FRAME) & df['class_id'].isin([1, 2, 3])]
print(after['display_track_id'].value_counts().sort_index().to_string())

In [ ]:
# ── Download ──────────────────────────────────────────────────────────────────
from google.colab import files as colab_files
colab_files.download(OUTPUT_CSV)